# 01 - Exploratory Analysis: COMPAS Recidivism Dataset

**Dataset:** COMPAS Recidivism Risk Score Data (ProPublica)  
**Objetivo:** Analisar a estrutura, qualidade e distribuições do dataset para informar decisões de sanitização e anonimização.

**Objetivo de divulgação:** Permitir a análise de possíveis disparidades raciais nos scores de risco de reincidência (COMPAS), por género e faixa etária, sem identificar os indivíduos avaliados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

df = pd.read_csv('../data/original/compas-scores-raw.csv')
print(f"Linhas: {len(df)}, Colunas: {len(df.columns)}")
print(f"Pessoas únicas (Person_ID): {df['Person_ID'].nunique()}")
df.head()

## 1. Estrutura do Dataset

Cada pessoa tem **3 linhas** (uma por tipo de score: Risk of Violence, Risk of Recidivism, Risk of Failure to Appear).

In [ ]:
# Tipos de dados e colunas
print("=== Colunas e Tipos ===")
print(df.dtypes)
print(f"\n=== Registos por pessoa ===")
vc = df['Person_ID'].value_counts()
print(f"  3 registos: {(vc==3).sum()} pessoas")
print(f"  6 registos: {(vc==6).sum()} pessoas")
print(f"  Outros: {(~vc.isin([3,6])).sum()} pessoas")

## 2. Qualidade dos Dados - Valores Nulos e Constantes

In [ ]:
# Valores nulos
print("=== Valores Nulos ===")
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
print(pd.DataFrame({'nulls': nulls, '%': nulls_pct}))

print("\n=== Colunas com 1 só valor (inúteis) ===")
for col in df.columns:
    if df[col].nunique() <= 1:
        print(f"  {col}: {df[col].unique()}")

print("\n=== Valores únicos por coluna ===")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} únicos")

## 3. Distribuição dos Atributos Categóricos

Analisar os valores de cada atributo categórico que vamos manter.

In [ ]:
cat_cols = ['Sex_Code_Text', 'Ethnic_Code_Text', 'MaritalStatus', 'Language',
            'LegalStatus', 'CustodyStatus', 'Agency_Text', 'RecSupervisionLevelText']

for col in cat_cols:
    print(f"\n=== {col} ({df[col].nunique()} valores) ===")
    vc = df[col].value_counts()
    vc_pct = (vc / len(df) * 100).round(2)
    print(pd.DataFrame({'count': vc, '%': vc_pct}).to_string())

## 4. Distribuição dos Scores (Sensitive Attributes)

`RawScore` e `DecileScore` são os atributos sensitive. Precisam de ter muitos valores distintos para l-Diversity.

In [ ]:
# Estatísticas dos scores por tipo
for score_type in ['Risk of Recidivism', 'Risk of Violence', 'Risk of Failure to Appear']:
    subset = df[df['DisplayText'] == score_type]
    print(f"\n=== {score_type} ===")
    print(f"  RawScore: {subset['RawScore'].nunique()} valores únicos")
    print(subset['RawScore'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2))
    print(f"\n  DecileScore:")
    print(subset['DecileScore'].value_counts().sort_index())

## 5. Análise de Idade (derivada de DateOfBirth)

Converter `DateOfBirth` para idade e analisar a distribuição.

In [ ]:
# Calcular idade a partir de DateOfBirth e Screening_Date
df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'])
df['Screening_Date'] = pd.to_datetime(df['Screening_Date'])
df['Age'] = ((df['Screening_Date'] - df['DateOfBirth']).dt.days / 365.25).astype(int)

print("=== Distribuição de Idade ===")
print(df['Age'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1))
print(f"\nValores únicos: {df['Age'].nunique()}")

# Distribuição por faixas
bins = [0, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 100]
labels = ['<20', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
print("\nDistribuição por faixa etária:")
print(df['Age_Group'].value_counts().sort_index())

## 6. Análise do Screening_Date

In [ ]:
print("=== Screening_Date ===")
print(f"  Min: {df['Screening_Date'].min()}")
print(f"  Max: {df['Screening_Date'].max()}")

print("\nDistribuição por ano:")
print(df['Screening_Date'].dt.year.value_counts().sort_index())

## 7. Redundâncias Confirmadas

Verificar que `ScoreText` é derivável de `DecileScore` e que `RecSupervisionLevel` é equivalente a `RecSupervisionLevelText`.

In [ ]:
print("=== ScoreText vs DecileScore ===")
print(df.groupby('ScoreText')['DecileScore'].agg(['min','max']))

print("\n=== RecSupervisionLevel vs RecSupervisionLevelText ===")
print(df[['RecSupervisionLevel','RecSupervisionLevelText']].drop_duplicates().sort_values('RecSupervisionLevel'))

## 8. Estatística do Objetivo (referência original)

Média do `DecileScore` de Recidivism por `Ethnic_Code_Text` × `Sex_Code_Text`.  
Esta tabela será comparada com o dataset anonimizado para avaliar a preservação de utilidade.

In [ ]:
# Filtrar apenas Risk of Recidivism
recid = df[df['DisplayText'] == 'Risk of Recidivism'].copy()

obj = recid.groupby(['Ethnic_Code_Text', 'Sex_Code_Text']).agg(
    mean_DecileScore=('DecileScore', 'mean'),
    mean_RawScore=('RawScore', 'mean'),
    count=('DecileScore', 'size')
).round(2)

print("=== Estatística do Objetivo - Dataset Original ===")
print(obj.to_string())

# Guardar para comparação futura
os.makedirs('../results', exist_ok=True)
obj.to_csv('../results/objetivo_original.csv')

## 9. Visualizações

In [ ]:
# 9.1 Distribuição de Idade
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Age'], bins=range(15, 85, 2), edgecolor='black', alpha=0.7)
ax.set_xlabel('Idade')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Idade')
plt.tight_layout()
plt.savefig('../results/dist_age.png', dpi=150)
plt.show()

In [ ]:
# 9.2 Distribuição por Etnia
fig, ax = plt.subplots(figsize=(10, 5))
ethnic_counts = recid['Ethnic_Code_Text'].value_counts()
ethnic_counts.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
ax.set_xlabel('Etnia')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição por Etnia')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../results/dist_ethnicity.png', dpi=150)
plt.show()

In [ ]:
# 9.3 DecileScore de Recidivism por Etnia (boxplot)
fig, ax = plt.subplots(figsize=(12, 6))
order = recid.groupby('Ethnic_Code_Text')['DecileScore'].median().sort_values(ascending=False).index
sns.boxplot(data=recid, x='Ethnic_Code_Text', y='DecileScore', order=order, ax=ax)
ax.set_xlabel('Etnia')
ax.set_ylabel('DecileScore (Recidivism)')
ax.set_title('DecileScore de Recidivism por Etnia')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../results/boxplot_decile_by_ethnicity.png', dpi=150)
plt.show()

In [ ]:
# 9.4 DecileScore de Recidivism por Etnia x Género (boxplot agrupado)
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=recid, x='Ethnic_Code_Text', y='DecileScore', hue='Sex_Code_Text', order=order, ax=ax)
ax.set_xlabel('Etnia')
ax.set_ylabel('DecileScore (Recidivism)')
ax.set_title('DecileScore de Recidivism por Etnia e Género')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Género')
plt.tight_layout()
plt.savefig('../results/boxplot_decile_by_ethnicity_gender.png', dpi=150)
plt.show()

In [ ]:
# 9.5 Correlação entre os 3 tipos de score (após pivotar temporariamente)
recid_pivot = recid[['Person_ID', 'RawScore']].copy()
violence = df[df['DisplayText'] == 'Risk of Violence'][['Person_ID', 'RawScore']].rename(columns={'RawScore': 'RawScore_Violence'})
fta = df[df['DisplayText'] == 'Risk of Failure to Appear'][['Person_ID', 'RawScore']].rename(columns={'RawScore': 'RawScore_FTA'})
recid_pivot = recid_pivot.rename(columns={'RawScore': 'RawScore_Recidivism'})

scores = recid_pivot.merge(violence, on='Person_ID').merge(fta, on='Person_ID')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(scores[['RawScore_Recidivism', 'RawScore_Violence', 'RawScore_FTA']].corr(),
            annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=ax, fmt='.2f')
ax.set_title('Correlação entre RawScores')
plt.tight_layout()
plt.savefig('../results/heatmap_score_correlation.png', dpi=150)
plt.show()

## 10. Resumo das Decisões

### Colunas a REMOVER
| Coluna | Razão |
|---|---|
| `Person_ID`, `AssessmentID`, `Case_ID` | Identifying (IDs internos) |
| `FirstName`, `LastName`, `MiddleName` | Identifying (nomes) |
| `DateOfBirth` | Identifying (convertido para `Age`) |
| `Screening_Date` | Convertido para `Screening_Year` |
| `ScaleSet_ID`, `Scale_ID` | IDs internos do sistema |
| `ScaleSet` | 96% um valor, pouca variação |
| `AssessmentReason` | Só 1 valor ("Intake") |
| `IsCompleted`, `IsDeleted` | Só 1 valor cada |
| `AssessmentType` | Metadata |
| `DisplayText` | Usado para pivotar, removido depois |
| `ScoreText` | Redundante (derivável de DecileScore) |
| `RecSupervisionLevel` | Redundante com `RecSupervisionLevelText` |

### Colunas a MANTER (após pivotar)
| Coluna | Classificação | Valores únicos |
|---|---|---|
| `Age` (derivado) | Quasi-identifying | contínuo |
| `Sex_Code_Text` | Quasi-identifying | 2 |
| `Ethnic_Code_Text` | Quasi-identifying | 9 |
| `MaritalStatus` | Quasi-identifying | 7 |
| `Language` | Quasi-identifying | 2 |
| `LegalStatus` | Quasi-identifying | 7 |
| `CustodyStatus` | Quasi-identifying | 6 |
| `Screening_Year` (derivado) | Quasi-identifying | ~3 |
| `Agency_Text` | Insensitive | 4 |
| `RecSupervisionLevelText` | Sensitive | 4 |
| `RawScore_Recidivism` | Sensitive | contínuo |
| `RawScore_Violence` | Sensitive | contínuo |
| `RawScore_FTA` | Sensitive | contínuo |
| `DecileScore_Recidivism` | Sensitive | 1-10 |
| `DecileScore_Violence` | Sensitive | 1-10 |
| `DecileScore_FTA` | Sensitive | 1-10 |

### Próximo passo
Sanitizar e pivotar o dataset no notebook `02_sanitize.ipynb`.